# 📝 Day 6 Assignments — Rate Limiting, Versioning & Docs

All tests run with `TestClient`.


In [ ]:
!pip install fastapi uvicorn httpx slowapi


## Task 1 — Rate Limit `/ping`

**Problem:** Set up a `Limiter(key_func=get_remote_address)`, hook it into the app, and add a `/ping` endpoint limited to **3 requests per minute**. Call it 4 times with `TestClient`; the 4th must return **429**.

**Expected output:**
```
req 1 -> 200
req 2 -> 200
req 3 -> 200
req 4 -> 429
```

💡 **Hint:** the route function must take `request: Request` as the first parameter, or slowapi can't extract the client key.


In [ ]:
from fastapi import FastAPI, Request
from fastapi.testclient import TestClient
from slowapi import Limiter, _rate_limit_exceeded_handler
from slowapi.errors import RateLimitExceeded
from slowapi.util import get_remote_address

# TODO: build limiter, app, hook them up, add /ping with @limiter.limit("3/minute")

client = TestClient(app)
for i in range(4):
    r = client.get("/ping")
    print(f"req {i + 1} -> {r.status_code}")


## Task 2 — v1 vs v2 Routers

**Problem:** Build two routers:
- `v1 = APIRouter(prefix="/v1", tags=["v1"])` — `/v1/items` returns `[{"id": 1, "name": "pen"}]`
- `v2 = APIRouter(prefix="/v2", tags=["v2"])` — `/v2/items` returns enriched dicts with `id`, `name`, `price`, `stock`

Include both on the app. Verify with `TestClient`.

**Expected output:**
```
v1: [{"id": 1, "name": "pen"}]
v2: [{"id": 1, "name": "pen", "price": 2.5, "stock": 100}]
```

💡 **Hint:** `app.include_router(v1)` after defining the routes on `v1`.


In [ ]:
from fastapi import FastAPI, APIRouter
from fastapi.testclient import TestClient

app = FastAPI()

# TODO: define v1 + v2 routers, attach /items on each, include them on app

client = TestClient(app)
print("v1:", client.get("/v1/items").json())
print("v2:", client.get("/v2/items").json())


## Task 3 — Customize OpenAPI Metadata

**Problem:** Create a FastAPI app with `title`, `description`, and `version` set. Add two endpoints (`/users` tagged `users`, `/items` tagged `items`). Print `client.get("/openapi.json").json()["info"]` and verify your custom values are there.

**Expected output:**
```
{"title": "My Bookshop", "description": "...", "version": "2.1.0"}
```

💡 **Hint:** `FastAPI(title=..., description=..., version=...)`.


In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient

# TODO: app = FastAPI(title=..., description=..., version=...)
# TODO: add /users (tags=["users"]) and /items (tags=["items"])

client = TestClient(app)
print(client.get("/openapi.json").json()["info"])


## Task 4 — Deprecation + Summaries

**Problem:**
1. Mark one endpoint with `deprecated=True`
2. Add `summary` and `description` to (at least) two endpoints
3. Verify by inspecting `/openapi.json`

**Expected output (snippet):**
```
/old GET deprecated=True summary="Use /new"
/new GET summary="Replacement" description="..."
```

💡 **Hint:** pass `summary=...`, `description=...`, `deprecated=True` to the route decorator.


In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient

app = FastAPI()

# TODO: add /old (deprecated=True, summary=...) and /new (summary=..., description=...)

client = TestClient(app)
paths = client.get("/openapi.json").json()["paths"]
for path, methods in paths.items():
    for method, op in methods.items():
        print(f"{path} {method.upper()} deprecated={op.get('deprecated')} summary={op.get('summary')!r}")


## 🎁 Bonus — Different Limits for Authenticated vs Anonymous

**Problem:** Combine Day 5 (JWT) and Day 6 (slowapi). Give authenticated users a higher rate limit than anonymous ones.

Approach:
1. Write a custom `key_func(request)` that:
   - Reads `Authorization: Bearer <token>` from the request headers
   - If valid → return the username (so the limit is per-user)
   - If missing/invalid → return the client IP (so anonymous users are limited by IP)
2. Use **two** decorators, or different routes, where authenticated paths allow `"100/minute"` and anonymous paths allow `"5/minute"`.

💡 **Hint:** `Limiter(key_func=my_key_func)`. Inside `my_key_func`, you have `request.headers.get("authorization")` — try to decode; on failure, fall back to `get_remote_address(request)`.


In [ ]:
# TODO: implement and demo the bonus with TestClient


---

✅ Final assignment of the backend section. You can now build a production-grade FastAPI service: validated, authenticated, rate-limited, versioned, and well-documented.
